# P110 — Una revisión sobre adaptación a la deriva de concepto

## 1. Título y paper

**Paper:** *A Survey on Concept Drift Adaptation*  
**Autoría:** João Gama, Indrė Žliobaitė, Albert Bifet, Mykola Pechenizkiy, Abdelhamid Bouchachia  
**Año y venue:** 2014 · ACM Computing Surveys, 46(4), 1–37  
**Nivel:** L3 · **Motor:** `deriva`  
**Ficha completa:** [`P110_deriva`](../../papers/foundational/P110_deriva/README.md)

**Hito:** Ordena el problema de que el mundo cambie después de entrenar, y separa detectar de adaptarse.

- [doi:10.1145/2523813](https://doi.org/10.1145/2523813)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un modelo se entrena con datos de un momento y se despliega sobre un flujo que cambia. La relación entre entradas y etiquetas puede cambiar sin que cambien las entradas, así que vigilar la distribución de entrada no basta y el modelo se degrada en silencio.
2. Ejecutar una implementación mínima de la propuesta: Una taxonomía de tipos de deriva —abrupta, gradual, incremental, recurrente— y de estrategias: detectores estadísticos sobre la tasa de error, ventanas adaptativas, conjuntos con reemplazo de miembros y reentrenamiento programado.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P76
- P82


## 4. Intuición

El modelo no se ha tocado. Los datos de entrada tienen el mismo aspecto de siempre. Y la exactitud se ha derrumbado. Lo que cambió no es el modelo ni las entradas: es la **relación** entre entradas y etiquetas.


## 5. Concepto mínimo

```text
Deriva de datos     : cambia P(x)        ← se detecta mirando las entradas
Deriva de concepto  : cambia P(y|x)      ← NO se detecta mirando las entradas

Detector tipo DDM: alarma cuando la tasa de error supera
                   su mínimo histórico + 3 desviaciones
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('deriva', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué le pasa a la exactitud tras la deriva?
2. ¿Detecta el cambio vigilar la distribución de entrada?
3. ¿Con cuánto retraso avisa el detector?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('deriva', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('deriva', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

La exactitud en ventana pasa de **1,0** a **0,4**. Y vigilar la distribución de entrada no habría detectado nada: es deriva de **concepto**, las entradas siguen igual. El detector avisa con un retraso de **1 muestra** en este caso limpio, y tras reentrenar la exactitud vuelve de 0,495 a 1,0.


## 10. Comentario pedagógico

Todo detector tiene retraso: es el precio de no dar falsas alarmas. Y detectar sirve de poco sin un procedimiento de reentrenamiento detrás — con datos etiquetados, validación y promoción. Esa cadena completa es lo que hace falta, y es la razón de que la monitorización sea una categoría propia en la rúbrica de [P112](../../papers/foundational/P112_ml_test_score/README.md).


## 11. Error o anti-patrón deliberado

Anti-patrón: vigilar solo la distribución de las entradas.


In [ ]:
print('La deriva de datos se ve mirando las entradas. La de CONCEPTO no.')
print('En la miniatura las entradas no cambian: cambia que etiqueta les corresponde.')
print('Un panel de distribuciones de entrada habria seguido en verde todo el tiempo.')

## 12. Corrección

Lo que sí detecta el cambio:


In [ ]:
r = run_paper_lab('deriva', seed=7)['result']
for c in r['curva_de_exactitud']:
    print(f"  instante {c['instante']:>4}  exactitud en ventana {c['exactitud_ventana']}")
print()
print('deriva real en   :', r['instante_de_la_deriva'])
print('alarma en        :', r['alarma_del_detector_en'], '| retraso', r['retraso_de_deteccion'])
print('tras la alarma   :', r['tras_la_alarma'])

## 13. Desafío guiado

Explica por qué todo detector tiene retraso y qué se compra a cambio de aumentarlo.


In [ ]:
r = run_paper_lab('deriva', seed=3)['result']
show(r)

## 14. Desafío autónomo

Elige un modelo tuyo en producción y define qué señal vigilarías para detectar deriva de concepto sin esperar a las etiquetas verdaderas. Documenta su retraso esperado.


## 15. Evidencia de aprendizaje

Guarda la curva de exactitud con el instante de la deriva y de la alarma, y tu plan de reentrenamiento.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P110_deriva/README.md) · evaluación formal: [`assessments/papers/P110_deriva.md`](../../assessments/papers/P110_deriva.md)


## 16. Cierre

El modelo ya se vigila. Ahora la pregunta incómoda: qué proporción del sistema es realmente el modelo.


## 17. Conexión con el siguiente hito

- P112

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
